# 01 · Análise Exploratória de Dados (AED)

**Projeto:** Cronos · Challenge FIAP 2026 com Locaweb  
**Autor(es):** Ana Beatriz Costa de Oliveira · Hygor Abrantes · Igor Vignola  
**Data de criação:** 20/05/2026  
**Última atualização:** 20/05/2026

## Objetivo

Explorar o LWDATASET (122.543 incidentes, 19 campos) para identificar padrões, sazonalidades e
distribuições, gerar os achados que alimentam o PPT da Sprint 2 (6-8 slides de AED) e justificar
as escolhas do plano de modelagem. Endereçar diretamente os pedidos da mentoria com a Locaweb:
anomalia de setembro/2025, sazonalidade real (feriado/fim-de-semana), padrão de cascata P4/P5 → P3/P2,
regra de KPI considerando só incidente pai, e especialização por grupo designado.

## Dependências

- Python 3.11+
- `pandas`, `numpy`, `matplotlib`, `seaborn`, `plotly`
- `holidays` (feriados BR)
- `openpyxl` (leitura do xlsx)

## Entradas

- `assets/Materal LocalWeb/LW-DATASET.xlsx` (aba `Dataset Geral`)

## Saídas

- Gráficos exportados para `notebooks/figures/01_eda/`
- Achados consolidados na Seção 5 deste notebook
- Insights documentados em `context/sprints/02-arquitetura.md`

---

In [5]:
# Stdlib
import warnings
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import holidays

# Local (quando houver módulos do projeto)
# from src.features import build_temporal_features

# Configurações
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Setup

Caminhos do projeto e aplicação do estilo visual Cronos (paleta + tipografia + layout) tanto em
Matplotlib quanto em Plotly. Toda figura gerada neste notebook deve sair com a identidade visual
do produto pronta pra ir pro PPT sem retrabalho.

In [6]:
# --- Paths do projeto ---
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_PATH = REPO_ROOT / 'assets' / 'Materal LocalWeb' / 'LW-DATASET.xlsx'
FIGURES_DIR = NOTEBOOK_DIR / 'figures' / '01_eda'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT  : {REPO_ROOT}')
print(f'DATA_PATH  : {DATA_PATH}  (existe: {DATA_PATH.exists()})')
print(f'FIGURES_DIR: {FIGURES_DIR}')

REPO_ROOT  : c:\Users\igor.vignola\Documents\Personal\FIAP\Challenge-LocaWeb
DATA_PATH  : c:\Users\igor.vignola\Documents\Personal\FIAP\Challenge-LocaWeb\assets\Materal LocalWeb\LW-DATASET.xlsx  (existe: True)
FIGURES_DIR: c:\Users\igor.vignola\Documents\Personal\FIAP\Challenge-LocaWeb\notebooks\figures\01_eda


In [7]:
# --- Identidade visual Cronos (paleta + tipografia + layout) ---
CRONOS_COLORS = {
    'black':      '#000000',
    'dark_gray':  '#444444',
    'mid_gray':   '#888888',
    'light_gray': '#F7F7F7',
    'white':      '#FFFFFF',
    'accent':     '#2563EB',  # azul Cronos — destaque
    'danger':     '#DC2626',  # vermelho — perigo/critico
    'success':    '#16A34A',
    'warning':    '#D97706',
}


def setup_cronos_style():
    """Aplica o estilo visual do Cronos ao Matplotlib."""
    mpl.rcParams.update({
        'figure.figsize': (10, 5),
        'figure.dpi': 100,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.edgecolor': '#444444',
        'axes.linewidth': 0.8,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.titlesize': 16,
        'axes.titleweight': 'bold',
        'axes.titlelocation': 'left',
        'axes.titlepad': 16,
        'axes.labelsize': 12,
        'axes.labelcolor': '#444444',
        'axes.grid': True,
        'axes.axisbelow': True,
        'grid.color': '#E5E5E5',
        'grid.linewidth': 0.5,
        'xtick.color': '#888888',
        'ytick.color': '#888888',
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'font.family': 'sans-serif',
        'font.sans-serif': ['Sora', 'Inter', 'DejaVu Sans'],
        'font.size': 11,
        'legend.frameon': False,
        'legend.fontsize': 11,
    })
    plt.rcParams['axes.grid.axis'] = 'y'


setup_cronos_style()

# Template Plotly equivalente
CRONOS_TEMPLATE = go.layout.Template(
    layout=dict(
        font=dict(family='Sora, Inter, sans-serif', size=12, color='#444444'),
        title=dict(font=dict(size=18, color='#000000'), x=0.02, xanchor='left'),
        paper_bgcolor='white',
        plot_bgcolor='white',
        xaxis=dict(
            showgrid=False,
            linecolor='#444444',
            tickcolor='#888888',
            tickfont=dict(size=11, color='#888888'),
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor='#E5E5E5',
            gridwidth=0.5,
            linecolor='#444444',
            tickcolor='#888888',
            tickfont=dict(size=11, color='#888888'),
        ),
        colorway=['#2563EB', '#000000', '#888888', '#DC2626', '#16A34A', '#D97706'],
    )
)
pio.templates['cronos'] = CRONOS_TEMPLATE
pio.templates.default = 'cronos'

print('Estilo Cronos aplicado (Matplotlib + Plotly).')

Estilo Cronos aplicado (Matplotlib + Plotly).


## 2. Carga dos dados

Apenas leitura do xlsx. Sem transformação aqui.

In [8]:
df = pd.read_excel(DATA_PATH, sheet_name='Dataset Geral')
print(f'Linhas: {len(df):,} | Colunas: {df.shape[1]}')

Linhas: 122,543 | Colunas: 19


## 3. Visão geral

Validação obrigatória antes de qualquer análise: amostra, tipos, estatísticas e faltantes.
Esta seção alimenta o **primeiro slide da AED no PPT** (Visão geral do dataset).

In [9]:
df.head()

,Número,Prioridade,Produto,Categoria,Subcategoria,Grupo designado,Item de configuração,Aberto,Resolvido,Encerrado,Duração,Código de fechamento,Descrição resumida,Solução,Aberto por,Incidente Pai,Status,Entrou para KPI?,KPI Violado?
0,INC8654273,3 - Média,NaN,NaN,NaN,Team14,IC00001,2025-12-31 23:45:18,NaT,2025-12-31 23:45:32,14,NaN,Problem: Apache Busy Workers,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
1,INC8654270,4 - Baixa,NaN,NaN,NaN,Team14,IC00002,2025-12-31 23:39:36,NaT,2025-12-31 23:43:05,209,NaN,Problem: Check Application Monitoring,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
2,INC8654264,4 - Baixa,NaN,NaN,NaN,Team14,NaN,2025-12-31 23:23:10,NaT,2025-12-31 23:25:00,110,NaN,Problem: Alarm Application Monitoring database...,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
3,INC8654263,4 - Baixa,NaN,NaN,NaN,Team14,NaN,2025-12-31 23:23:07,NaT,2025-12-31 23:24:57,110,NaN,Problem: Alarm Application Monitoring coupons ...,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN
4,INC8654262,4 - Baixa,NaN,NaN,NaN,Team14,IC00003,2025-12-31 23:23:05,NaT,2025-12-31 23:23:47,42,NaN,Problem: Check Application Monitoring,NaN,Monitoramento,NaN,Sem Intervenção,NAO,NaN


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122543 entries, 0 to 122542
Data columns (total 19 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   Número                122543 non-null  object        
 1   Prioridade            122543 non-null  object        
 2   Produto               44608 non-null   object        
 3   Categoria             44822 non-null   object        
 4   Subcategoria          44823 non-null   object        
 5   Grupo designado       122543 non-null  object        
 6   Item de configuração  120763 non-null  object        
 7   Aberto                122543 non-null  datetime64[ns]
 8   Resolvido             40241 non-null   datetime64[ns]
 9   Encerrado             122543 non-null  datetime64[ns]
 10  Duração               122543 non-null  int64         
 11  Código de fechamento  40804 non-null   object        
 12  Descrição resumida    122543 non-null  object        
 13 

In [11]:
df.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Número,122543,122543,INC7227672,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Prioridade,122543,5,4 - Baixa,64828,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Produto,44608,51,lhco,12835,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Categoria,44822,141,cat71,7335,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Subcategoria,44823,447,sub7,4668,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Grupo designado,122543,17,Team14,92775,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Item de configuração,120763,9171,IC00014,6069,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Aberto,122543,NaN,NaN,NaN,2025-09-18 22:10:35.012746752,2023-01-02 20:19:58,2025-09-02 10:30:35,2025-10-14 00:11:49,2025-11-26 18:11:42.500000,2025-12-31 23:45:18,NaN
Resolvido,40241,NaN,NaN,NaN,2025-06-30 07:35:15.915683328,2024-12-27 00:50:52,2025-03-26 08:17:00,2025-07-01 13:51:22,2025-10-01 16:07:13,2025-12-31 19:36:58,NaN
Encerrado,122543,NaN,NaN,NaN,2025-09-22 18:42:48.700783104,2025-01-01 00:10:45,2025-09-03 18:35:03.500000,2025-10-15 23:55:36,2025-11-28 01:53:03,2025-12-31 23:45:32,NaN


In [12]:
df.isna().sum().sort_values(ascending=False)

Incidente Pai           107416
Solução                 107243
KPI Violado?             96943
Resolvido                82302
Código de fechamento     81739
Produto                  77935
Categoria                77721
Subcategoria             77720
Item de configuração      1780
Prioridade                   0
Número                       0
Duração                      0
Encerrado                    0
Aberto                       0
Grupo designado              0
Aberto por                   0
Descrição resumida           0
Status                       0
Entrou para KPI?             0
dtype: int64

## 4. Análises

Cada subseção segue o tripé **Pergunta → Código → Achado**. Ordem definitiva confirmada
com Igor após a Visão geral.

Subseções previstas:
- 4.1 Distribuição temporal (mensal/diária)
- 4.2 Anomalia de setembro/2025 — investigação aprofundada
- 4.3 Sazonalidade (feriado vs. fim-de-semana vs. dia útil)
- 4.4 Distribuição por prioridade + status + 'Aberto por'
- 4.5 Top produtos/categorias críticos
- 4.6 Análise de OLA (somente pais que entram no KPI)
- 4.7 Padrão de cascata P4/P5 → P3/P2
- 4.8 Especialização por grupo designado

## 5. Conclusões e próximos passos

_A preencher ao final, depois das análises._